# GPT 펑션 콜링(Function Calling) 예제

모델이 우리가 정의한 함수(도구)를 스스로 골라 호출하는 **Function Calling** 예제입니다.

동작 흐름 (Responses API 기준):
1. `tools` 파라미터로 함수 스키마를 전달
2. 모델이 필요하다고 판단하면 응답에 `function_call` 아이템을 반환
3. 우리가 실제 파이썬 함수를 실행하고, 결과를 `function_call_output`으로 다시 전달
4. 모델이 함수 결과를 반영한 최종 답변 생성 (필요하면 2~3을 반복)

In [8]:
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = "gpt-5-nano"

## 1. 도구(함수) 3개 정의

실습용으로 실제 API 대신 목(mock) 데이터를 반환하는 함수 3개를 만듭니다.

| 함수 | 역할 |
|---|---|
| `get_weather` | 도시의 현재 날씨 조회 |
| `get_exchange_rate` | 환율 조회 |
| `calculate` | 사칙연산 계산기 |

In [9]:
import ast
import operator


def get_weather(city: str) -> dict:
    """도시의 현재 날씨를 조회합니다 (목 데이터)."""
    mock_data = {
        "서울": {"temp_c": 29, "condition": "맑음", "humidity": 55},
        "부산": {"temp_c": 27, "condition": "구름 조금", "humidity": 70},
        "제주": {"temp_c": 26, "condition": "비", "humidity": 85},
    }
    return {"city": city, **mock_data.get(city, {"temp_c": 25, "condition": "정보 없음", "humidity": 60})}


def get_exchange_rate(base: str, target: str) -> dict:
    """두 통화 간 환율을 조회합니다 (목 데이터)."""
    mock_rates = {("USD", "KRW"): 1385.50, ("EUR", "KRW"): 1512.30, ("JPY", "KRW"): 9.41}
    rate = mock_rates.get((base.upper(), target.upper()))
    if rate is None:
        return {"error": f"{base}/{target} 환율 정보 없음"}
    return {"base": base.upper(), "target": target.upper(), "rate": rate}


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv}


def calculate(expression: str) -> dict:
    """사칙연산 수식 문자열을 안전하게 계산합니다 (eval 미사용)."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -_eval(node.operand)
        raise ValueError(f"지원하지 않는 수식: {expression}")
    result = _eval(ast.parse(expression, mode="eval").body)
    return {"expression": expression, "result": round(result, 4)}


# 함수 이름 → 실제 파이썬 함수 매핑
FUNCTIONS = {
    "get_weather": get_weather,
    "get_exchange_rate": get_exchange_rate,
    "calculate": calculate,
}

## 2. 모델에 전달할 도구 스키마

Responses API의 함수 도구는 `type: "function"` + JSON Schema 형식의 `parameters`로 정의합니다.
`strict: True`를 주면 모델이 스키마를 정확히 지키도록 강제됩니다.

In [10]:
TOOLS = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "도시의 현재 날씨(기온, 상태, 습도)를 조회합니다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "도시 이름 (예: 서울, 부산, 제주)"}
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_exchange_rate",
        "description": "기준 통화(base)에서 대상 통화(target)로의 환율을 조회합니다.",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "base": {"type": "string", "description": "기준 통화 코드 (예: USD)"},
                "target": {"type": "string", "description": "대상 통화 코드 (예: KRW)"}
            },
            "required": ["base", "target"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "calculate",
        "description": "사칙연산 수식 문자열을 계산합니다. 예: '100 * 1385.5'",
        "strict": True,
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "계산할 수식 (숫자와 + - * / 만 사용)"}
            },
            "required": ["expression"],
            "additionalProperties": False,
        },
    },
]

## 3. 에이전트 루프

모델이 `function_call`을 반환하는 동안 계속 함수를 실행해서 결과를 돌려주고,
함수 호출이 더 이상 없으면 최종 답변을 반환하는 루프입니다.

In [11]:
def run_conversation(question: str, max_rounds: int = 5) -> str:
    print(f"💬 질문: {question}\n")
    input_list = [{"role": "user", "content": question}]
    total_calls = 0

    for _ in range(max_rounds):
        response = client.responses.create(model=MODEL, input=input_list, tools=TOOLS)

        # 응답 아이템(reasoning, function_call 등)을 대화 기록에 그대로 누적
        input_list += response.output

        function_calls = [item for item in response.output if item.type == "function_call"]
        if not function_calls:
            print(f"\n✅ 총 함수 호출 횟수: {total_calls}회")
            print(f"\n🤖 최종 답변:\n{response.output_text}")
            return response.output_text

        # 모델이 요청한 함수들을 실행하고 결과를 돌려준다
        for call in function_calls:
            args = json.loads(call.arguments)
            result = FUNCTIONS[call.name](**args)
            total_calls += 1
            print(f"  🔧 [{total_calls}] {call.name}({args})\n      → {result}")
            input_list.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            })

    raise RuntimeError("최대 라운드를 초과했습니다.")

## 예시 1 — 단일 함수 호출

날씨 질문 → `get_weather` 1회 호출

In [12]:
_ = run_conversation("서울 지금 날씨 어때?")

💬 질문: 서울 지금 날씨 어때?

  🔧 [1] get_weather({'city': '서울'})
      → {'city': '서울', 'temp_c': 29, 'condition': '맑음', 'humidity': 55}

✅ 총 함수 호출 횟수: 1회

🤖 최종 답변:
서울은 현재 **맑음**, 기온은 **29°C**, 습도는 **55%**입니다.


## 예시 2 — 한 번의 질문으로 함수 3회 호출

날씨 2개 도시 + 환율까지 물어보면 모델이 필요한 함수를 알아서 3번 호출합니다.

In [7]:
_ = run_conversation("서울이랑 부산 날씨 알려주고, 지금 1달러가 몇 원인지도 알려줘.")

💬 질문: 서울이랑 부산 날씨 알려주고, 지금 1달러가 몇 원인지도 알려줘.

  🔧 [1] get_weather({'city': '서울'})
      → {'city': '서울', 'temp_c': 29, 'condition': '맑음', 'humidity': 55}
  🔧 [2] get_weather({'city': '부산'})
      → {'city': '부산', 'temp_c': 27, 'condition': '구름 조금', 'humidity': 70}
  🔧 [3] get_exchange_rate({'base': 'USD', 'target': 'KRW'})
      → {'base': 'USD', 'target': 'KRW', 'rate': 1385.5}

✅ 총 함수 호출 횟수: 3회

🤖 최종 답변:
현재 정보입니다.

- 서울: 29°C, 맑음, 습도 55%
- 부산: 27°C, 구름 조금, 습도 70%
- 환율: 1달러 = 1,385.5원


## 예시 3 — 연쇄 호출 (앞 함수의 결과를 다음 함수가 사용)

환율을 먼저 조회한 뒤, 그 결과값으로 계산기를 호출해야 답할 수 있는 질문입니다.
모델이 `get_exchange_rate` → `calculate` 순서로 스스로 연쇄 호출합니다.

In [7]:
_ = run_conversation("350달러를 원화로 환전하면 정확히 얼마야? 환율 조회하고 계산기로 계산해줘.")

💬 질문: 350달러를 원화로 환전하면 정확히 얼마야? 환율 조회하고 계산기로 계산해줘.



  🔧 [1] get_exchange_rate({'base': 'USD', 'target': 'KRW'})
      → {'base': 'USD', 'target': 'KRW', 'rate': 1385.5}


  🔧 [2] calculate({'expression': '350 * 1385.5'})
      → {'expression': '350 * 1385.5', 'result': 484925.0}



✅ 총 함수 호출 횟수: 2회

🤖 최종 답변:
조회한 환율: **1 USD = 1,385.5 KRW**

계산: **350 × 1,385.5 = 484,925**

따라서 **350달러는 484,925원**입니다.
